# IGS SINEX Station Coordinate Downloader

This notebook downloads and parses IGS (International GNSS Service) reference frame station coordinates from SINEX files.

## Step 1: Install Required Packages

Run this cell first to install the required packages.

In [ ]:
# Install required packages
!pip install requests compress

## Step 2: Import Libraries

In [ ]:
import requests
import compress
from pathlib import Path
import math

## Step 3: Download and Decompress SINEX File

Change the `gps_week` variable to download data for a different week.

In [ ]:
def download_igs_sinex(gps_week, output_dir='.'):
    """
    Download and decompress IGS SINEX file for a given GPS week
    
    Args:
        gps_week: GPS week number as string (e.g., "2340")
        output_dir: Directory to save the file (default: current directory)
    
    Returns:
        Path to the downloaded SINEX file, or None if failed
    """
    
    # Construct URL
    base_url = "https://cddis.gsfc.nasa.gov/archive/gnss/products"
    filename = f"igs{gps_week}.snx.Z"
    url = f"{base_url}/{gps_week}/{filename}"
    
    print(f"Downloading IGS SINEX file")
    print(f"GPS Week: {gps_week}")
    print(f"URL: {url}\n")
    
    try:
        # Download file
        print("Downloading...")
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        print(f"✓ Downloaded {len(response.content)} bytes")
        
        # Decompress using compress module
        print("Decompressing...")
        import io
        # compress module expects file-like objects
        compressed_file = io.BytesIO(response.content)
        decompressed_file = io.BytesIO()
        compress.uncompress(compressed_file, decompressed_file)
        decompressed = decompressed_file.getvalue()
        print(f"✓ Decompressed to {len(decompressed)} bytes")
        
        # Save
        output_path = Path(output_dir) / f"igs{gps_week}.snx"
        with open(output_path, 'wb') as f:
            f.write(decompressed)
        
        print(f"\n✓ SUCCESS! File saved to: {output_path.absolute()}")
        print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")
        
        return output_path
        
    except requests.exceptions.HTTPError as e:
        print(f"\n✗ HTTP Error: {e}")
        print(f"  The file might not exist for GPS week {gps_week}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"\n✗ Download failed: {e}")
        return None
    except Exception as e:
        print(f"\n✗ Error: {e}")
        import traceback
        traceback.print_exc()
        return None

# Download SINEX file for GPS week 2340 (October 2024)
# Change this to your desired GPS week
gps_week = "2340"

sinex_file = download_igs_sinex(gps_week)

## Step 4: Preview the SINEX File

Let's look at the first 30 lines of the file.

In [ ]:
if sinex_file and sinex_file.exists():
    print("First 30 lines of SINEX file:\n")
    print("=" * 80)
    
    with open(sinex_file, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if i >= 30:
                break
            print(line.rstrip())
else:
    print("No file to preview. Please run the download cell above first.")

## Step 5: Parse Station Coordinates

Extract XYZ coordinates for all stations in the SINEX file.

In [ ]:
def parse_sinex_coordinates(sinex_file):
    """
    Parse SINEX file to extract station coordinates
    
    Args:
        sinex_file: Path to SINEX file
    
    Returns:
        Dictionary of station codes and their XYZ coordinates
    """
    stations = {}
    in_estimate_block = False
    
    print(f"Parsing {sinex_file}...")
    
    try:
        with open(sinex_file, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                # Start of solution estimate block
                if '+SOLUTION/ESTIMATE' in line:
                    in_estimate_block = True
                    continue
                
                # End of solution estimate block
                if '-SOLUTION/ESTIMATE' in line:
                    break
                
                # Parse coordinate lines
                if in_estimate_block and line.startswith(' '):
                    parts = line.split()
                    if len(parts) >= 9:
                        param_type = parts[1]  # STAX, STAY, STAZ
                        station_code = parts[2]
                        value = float(parts[8])
                        
                        if station_code not in stations:
                            stations[station_code] = {'X': 0, 'Y': 0, 'Z': 0}
                        
                        if param_type == 'STAX':
                            stations[station_code]['X'] = value
                        elif param_type == 'STAY':
                            stations[station_code]['Y'] = value
                        elif param_type == 'STAZ':
                            stations[station_code]['Z'] = value
        
        print(f"✓ Found {len(stations)} stations")
        return stations
        
    except Exception as e:
        print(f"✗ Error parsing file: {e}")
        return {}

# Parse the SINEX file
if sinex_file and sinex_file.exists():
    stations = parse_sinex_coordinates(sinex_file)
else:
    print("No file to parse. Please run the download cell first.")
    stations = {}

## Step 6: Convert XYZ to Latitude/Longitude/Altitude

In [ ]:
def xyz_to_lat_lon_alt(x, y, z):
    """
    Convert ECEF XYZ to Latitude, Longitude, Altitude (WGS84)
    
    Args:
        x, y, z: ECEF coordinates in meters
    
    Returns:
        (latitude, longitude, altitude) in degrees and meters
    """
    # WGS84 ellipsoid parameters
    a = 6378137.0  # semi-major axis
    f = 1 / 298.257223563  # flattening
    e2 = 2 * f - f * f  # first eccentricity squared
    
    # Calculate longitude
    lon = math.atan2(y, x)
    
    # Calculate latitude iteratively
    p = math.sqrt(x * x + y * y)
    lat = math.atan2(z, p * (1 - e2))
    
    for _ in range(5):  # iterate to converge
        N = a / math.sqrt(1 - e2 * math.sin(lat) ** 2)
        lat = math.atan2(z + e2 * N * math.sin(lat), p)
    
    # Calculate altitude
    N = a / math.sqrt(1 - e2 * math.sin(lat) ** 2)
    alt = p / math.cos(lat) - N
    
    # Convert to degrees
    lat_deg = math.degrees(lat)
    lon_deg = math.degrees(lon)
    
    return lat_deg, lon_deg, alt

## Step 7: Display Results

Show the first 10 stations with their coordinates in both XYZ and Lat/Lon/Alt formats.

In [ ]:
if stations:
    print("\nFirst 10 stations (XYZ coordinates in meters):")
    print("=" * 80)
    
    for i, (code, coords) in enumerate(list(stations.items())[:10]):
        x, y, z = coords['X'], coords['Y'], coords['Z']
        lat, lon, alt = xyz_to_lat_lon_alt(x, y, z)
        
        print(f"\n{code}:")
        print(f"  XYZ: {x:14.4f}, {y:14.4f}, {z:14.4f}")
        print(f"  Lat/Lon/Alt: {lat:10.6f}°, {lon:10.6f}°, {alt:8.2f}m")
    
    print(f"\n{'=' * 80}")
    print(f"Total stations in file: {len(stations)}")
    print(f"{'=' * 80}")
else:
    print("No station data available. Please run the parsing cell above.")

## Step 8: Export to DataFrame (Optional)

Convert the station data to a pandas DataFrame for easier analysis.

In [ ]:
# Install pandas if needed
try:
    import pandas as pd
except ImportError:
    print("Installing pandas...")
    !pip install pandas
    import pandas as pd

if stations:
    # Create list of station data
    data = []
    for code, coords in stations.items():
        x, y, z = coords['X'], coords['Y'], coords['Z']
        lat, lon, alt = xyz_to_lat_lon_alt(x, y, z)
        data.append({
            'Station': code,
            'X': x,
            'Y': y,
            'Z': z,
            'Latitude': lat,
            'Longitude': lon,
            'Altitude': alt
        })
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    print(f"Created DataFrame with {len(df)} stations\n")
    print(df.head(10))
    
    # Optionally save to CSV
    # df.to_csv(f'igs_{gps_week}_stations.csv', index=False)
    # print(f"\nSaved to: igs_{gps_week}_stations.csv")
else:
    print("No station data to export.")

## Step 9: Search for Specific Stations (Optional)

Search for stations by code or location.

In [ ]:
def find_station(station_code):
    """Find and display information for a specific station"""
    if station_code in stations:
        coords = stations[station_code]
        x, y, z = coords['X'], coords['Y'], coords['Z']
        lat, lon, alt = xyz_to_lat_lon_alt(x, y, z)
        
        print(f"Station: {station_code}")
        print(f"  XYZ: {x:.4f}, {y:.4f}, {z:.4f}")
        print(f"  Lat/Lon/Alt: {lat:.6f}°, {lon:.6f}°, {alt:.2f}m")
    else:
        print(f"Station {station_code} not found in dataset")
        print(f"Total stations available: {len(stations)}")

# Example: Search for a station
# Uncomment and change the station code below
# find_station("ALIC")  # Alice Springs, Australia
# find_station("AREQ")  # Arequipa, Peru

## Notes

### GPS Week Calculator
To find the GPS week for a specific date: https://www.timeanddate.com/date/gps-week-calculator.html

### Example GPS Weeks
- Week 2345: December 2024
- Week 2340: October 2024  
- Week 2300: February 2024
- Week 2250: October 2023

### Data Source
- NASA CDDIS: https://cddis.gsfc.nasa.gov/archive/gnss/products/
- IGS Website: https://igs.org/

### Reference Frame
The coordinates are in the IGS20 reference frame (aligned with ITRF2020).